# FineWeb-Edu GPT — Analysis

Training now happens in `train.py`:

    uv run python projects/fineweb-edu/gpt/train.py

A decoder-only Transformer (grouped-query attention + RoPE + QK-norm) pretrained on
[FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) `sample-10BT`,
tokenized with the pretrained [LiquidAI/LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M)
subword tokenizer, optimized with Muon + AdamW and Cut Cross Entropy.

This notebook only loads the resulting checkpoint and metrics for analysis:
a data preview, training curves, and text generation.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import glob
import os
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import torch

from chimera.data import FineWebEduDataModule
from chimera.models import GPT

# datasets + tokenizer caches live on the big volume
os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")

RUN_DIR = "/mnt/ai/runs/fineweb-edu/gpt"
DATA_DIR = "/mnt/ai/data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Must match the model trained by train.py (its defaults).
SEQ_LEN = 2048
N_EMBD = 384
N_HEAD = 12
N_KV_HEAD = 3
N_LAYER = 6
TIE_EMBEDDING = True

## Data

Load FineWeb-Edu `sample-10BT` with the fixed **LiquidAI/LFM2.5-230M** tokenizer.
Documents are concatenated with `<|endoftext|>` appended after each, so the model
learns document boundaries instead of attending across unrelated pages.

In [ ]:
dm = FineWebEduDataModule(
    data_dir=DATA_DIR,
    name="sample-10BT",
    batch_size=8,
    seq_len=SEQ_LEN,
    tokenizer_backend="pretrained",  # LiquidAI/LFM2.5-230M
    add_eos=True,
    max_train_tokens=1_000_000_000,
    num_workers=4,
)
dm.prepare_data()
dm.setup("fit")

print(f"tokenizer={dm.pretrained_id}  vocab_size={dm.vocab_size}")
print(f"eos_token={dm.eos_token!r}  eos_id={dm.eos_id}")
print(f"train chunks={len(dm.train_dataset):,}  val chunks={len(dm.val_dataset):,}")

x, y = next(iter(dm.train_dataloader()))
print("sample:", repr(dm.decode(x[0][:64])))
# confirm the document separator is actually present in the stream
n_eos = int((x == dm.eos_id).sum())
print(f"<|endoftext|> tokens in one batch of {x.numel():,}: {n_eos}")

# most frequent tokens in one batch, shown as their decoded text
counts = Counter(x.flatten().tolist())
top = counts.most_common(20)
labels = [repr(dm.decode([tid])) for tid, _ in top]
values = [c for _, c in top]

plt.figure(figsize=(12, 4))
plt.bar(range(len(top)), values)
plt.title("Top-20 Tokens (one training batch)")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(range(len(top)), labels, rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Load checkpoint

In [ ]:
ckpt = torch.load(
    f"{RUN_DIR}/checkpoints/gpt.ckpt", map_location="cpu", weights_only=False
)
# train.py torch.compile()s the model, so state_dict keys are prefixed
# "model._orig_mod." (compiled) or "model." (eager) -- strip both.
model_state = {
    k.removeprefix("model.").removeprefix("_orig_mod."): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("model.")
}

model = GPT(
    vocab_size=dm.vocab_size,
    block_size=SEQ_LEN,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    n_layer=N_LAYER,
    tie_embedding=TIE_EMBEDDING,
)
model.load_state_dict(model_state)
model.to(DEVICE).eval()
print(
    f"loaded GPT ({sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params, "
    f"epoch {ckpt['epoch']}, step {ckpt['global_step']})"
)

## Training curves

Reads the CSV log written by `chimera.utils.build_run_loggers`. Validation is
logged under both `val/*` (during `fit`) and `test/*` (the final `trainer.test`).

In [ ]:
csv_path = sorted(glob.glob(f"{RUN_DIR}/csv/version_*/metrics.csv"))[-1]
metrics = pd.read_csv(csv_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Training Metrics")
for ax, key, title in zip(axes, ["loss", "bpt"], ["Loss (nats)", "Bits per Token"]):
    for stage, style in [
        ("train", {}),
        ("val", {"marker": "o", "ls": ""}),
        ("test", {"marker": "s", "ls": ""}),
    ]:
        col = f"{stage}/{key}"
        if col not in metrics.columns:
            continue
        d = metrics.dropna(subset=[col])
        if not len(d):
            continue
        ax.plot(d["step"], d[col], label=stage, **style)
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

## Text generation

In [ ]:
prompt = "The role of education in modern society is "
idx = torch.tensor([dm.tokenizer.encode(prompt)], device=DEVICE)

# Eager by default (no compile warmup). Pass compile=True for ~2x faster decode
# when generating repeatedly -- it costs a one-time ~10s compile on the first call.
generated = model.generate(idx, max_new_tokens=200, temperature=1)
print(dm.decode(generated[0].cpu()))